## Section 1 — Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, recall_score, precision_score, f1_score

import joblib

SEED = 42

In [ ]:
df = pd.read_csv('data/heart_cleveland_upload.csv')
print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Columns:", df.columns.tolist())
print("\nTarget distribution:")
print(df['condition'].value_counts())
print(df['condition'].value_counts(normalize=True).round(3))

In [ ]:
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nBasic statistics:")
df.describe()

## Section 2 — Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='condition', data=df, palette='Set2')
ax.set_title('Heart Disease Class Distribution')
ax.set_xticklabels(['No Disease', 'Disease'])
ax.set_xlabel('Condition')
ax.set_ylabel('Count')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=12)

total = len(df)
for i, val in enumerate(df['condition'].value_counts().sort_index()):
    print(f"{'No Disease' if i == 0 else 'Disease'}: {val} ({val/total*100:.1f}%)")

plt.tight_layout()
plt.show()

In [ ]:
continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feature in enumerate(continuous_features):
    ax = axes[i]
    ax.hist(df[df['condition'] == 0][feature], alpha=0.6, color='steelblue', label='No Disease', bins=20)
    ax.hist(df[df['condition'] == 1][feature], alpha=0.6, color='tomato', label='Disease', bins=20)
    ax.set_title(feature)
    ax.set_xlabel(feature)
    ax.set_ylabel('Count')
    ax.legend()

axes[5].axis('off')
axes[5].text(0.5, 0.5, 'Continuous features:\nOverlapping histograms\nby disease class',
             ha='center', va='center', fontsize=11, color='gray')

plt.suptitle('Continuous Feature Distributions by Disease Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feature in enumerate(categorical_features):
    ax = axes[i]
    sns.barplot(x=feature, y='condition', data=df, palette='coolwarm', ax=ax)
    ax.set_title(f'Disease Rate by {feature}')
    ax.set_xlabel(feature)
    ax.set_ylabel('Mean Disease Rate')
    ax.set_ylim(0, 1)

plt.suptitle('Categorical Feature Disease Rates', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Key correlations to note:
# cp       vs condition: strong positive — chest pain type is a direct symptom
# thalach  vs condition: negative — higher max heart rate associated with no disease
# exang    vs condition: positive — exercise-induced angina correlates with disease

In [ ]:
continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feature in enumerate(continuous_features):
    ax = axes[i]
    sns.boxplot(x='condition', y=feature, data=df, palette='Set3', ax=ax)
    ax.set_title(f'{feature} by Disease Class')
    ax.set_xticklabels(['No Disease', 'Disease'])
    ax.set_xlabel('Condition')

axes[5].axis('off')

plt.suptitle('Continuous Feature Distributions (Boxplots) by Disease Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Section 3 — Preprocessing and Feature Engineering

In [ ]:
X = df.drop(columns=['condition'])
y = df['condition']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature names:", X.columns.tolist())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape: ", y_test.shape)

print("\nClass distribution in training set:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).round(3))

print("\nClass distribution in test set:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True).round(3))

# stratify=y ensures both train and test sets mirror the original
# ~54% no disease / ~46% disease class split — critical for reliable evaluation

In [ ]:
scaler = StandardScaler()

# Fit ONLY on training data, then transform both sets
# Fitting on test data would cause data leakage — the model would have
# indirect knowledge of the test distribution before evaluation
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Why scaling matters for SVM:
# SVM finds the maximum-margin hyperplane using Euclidean distances.
# If 'chol' ranges 0–600 and 'fbs' ranges 0–1, chol dominates the
# distance computation regardless of predictive value. StandardScaler
# brings every feature to mean=0, std=1 so all features contribute equally.
#
# Logistic Regression also benefits — gradient descent converges faster
# on scaled features. Random Forest does not require scaling (threshold-
# based splits are scale-invariant) but we apply it here for consistency.

In [ ]:
print("X_train_scaled mean per feature (should be ~0):")
print(X_train_scaled.mean(axis=0).round(3))

print("\nX_train_scaled std per feature (should be ~1):")
print(X_train_scaled.std(axis=0).round(3))

print("\nX_test_scaled mean per feature (NOT necessarily 0):")
print(X_test_scaled.mean(axis=0).round(3))

# The test set mean will not be exactly 0 — this is correct and expected.
# The scaler was fitted on training data only. Applying those same
# mean/std parameters to the test set is intentional: it simulates
# how the model would scale unseen real-world patient data at inference time.

## Section 4 — Why SVM Works Well for Medical Classification

### The Core Idea: Maximum Margin Separation

SVM finds the hyperplane in feature space that maximally separates the two classes. Among all hyperplanes that correctly separate the classes, SVM selects the one with the **largest margin** — the maximum distance between the hyperplane and the nearest data points from each class. Those nearest points are called **support vectors**.

### The Optimization Problem

For binary classification with labels y in {-1, +1} and feature vectors x, SVM solves:

**Minimize:** (1/2) * ||w||²

**Subject to:** y_i * (w · x_i + b) >= 1 for all training points i

where w is the weight vector defining the hyperplane orientation and b is the bias term. The margin width equals 2 / ||w||, so minimizing ||w||² is equivalent to maximizing the margin.

The soft-margin extension adds a penalty parameter C that controls the tradeoff between maximizing the margin and allowing some misclassifications. High C = narrow margin, fewer training errors, risk of overfitting. Low C = wide margin, more training errors, better generalization.

### The Kernel Trick

Real medical data is rarely linearly separable. The kernel trick extends SVM to non-linear boundaries by implicitly mapping data into a higher-dimensional space without ever computing that mapping explicitly.

The **RBF (Radial Basis Function) kernel** computes:

K(x_i, x_j) = exp(-gamma * ||x_i - x_j||²)

This measures similarity between two patient records. Records that are close together in feature space have kernel value near 1. Records far apart have kernel value near 0. The gamma parameter controls how tightly each support vector influences the boundary — high gamma = tight local influence, low gamma = broad smooth influence.

### Why SVM Suits This Dataset Specifically

**High dimensionality relative to sample size:** The Cleveland dataset has 13 features and 303 samples. SVM with RBF kernel finds non-linear boundaries in 13-dimensional space while C prevents overfitting to the small sample.

**Robustness to outliers:** The decision boundary is determined only by support vectors — the points nearest the margin — not all 303 training records. Noisy or anomalous patient records that are far from the boundary have zero influence on where the boundary sits. Clinical datasets routinely contain measurement noise and data entry errors, making this property especially valuable.

**No probability calibration required for threshold tuning:** SVM with probability=True uses Platt scaling to produce well-calibrated probabilities, allowing us to tune the classification threshold in Section 6 to prioritize recall over precision — directly encoding the clinical preference for catching every sick patient even at the cost of some false alarms.

### The Three-Way Tradeoff This Model Must Navigate

| Decision | Clinical Meaning |
|---|---|
| False Negative (predict healthy, actually sick) | Patient goes home untreated — highest cost |
| False Positive (predict sick, actually healthy) | Unnecessary follow-up test — lower cost |
| Threshold tuning | Lower threshold → more FP, fewer FN → clinically preferable |

In [ ]:
# Illustrative only — actual model uses all 13 features
# Using thalach and oldpeak: two of the strongest continuous predictors

feature_names = X.columns.tolist()
thalach_idx = feature_names.index('thalach')
oldpeak_idx = feature_names.index('oldpeak')

X_2d = X_train_scaled[:, [thalach_idx, oldpeak_idx]]
y_2d = y_train.values

# Fit a linear SVM on just these 2 features for clean boundary visualization
svm_2d = SVC(kernel='linear', C=1.0, random_state=SEED)
svm_2d.fit(X_2d, y_2d)

# Build meshgrid over the 2D feature space
x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

Z = svm_2d.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))

# Decision boundary and margin lines
ax.contour(xx, yy, Z, levels=[-1, 0, 1],
           linestyles=['--', '-', '--'],
           colors=['gray', 'black', 'gray'],
           linewidths=[1.5, 2.5, 1.5])

# Fill the two class regions
ax.contourf(xx, yy, Z, levels=[-999, 0, 999],
            alpha=0.1, colors=['steelblue', 'tomato'])

# Scatter the training points
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                     c=y_2d, cmap='bwr', alpha=0.7,
                     edgecolors='k', linewidths=0.5, s=60,
                     label='Training points')

# Circle the support vectors
sv = svm_2d.support_vectors_
ax.scatter(sv[:, 0], sv[:, 1],
           s=200, facecolors='none', edgecolors='black',
           linewidths=2.0, label='Support Vectors', zorder=5)

ax.set_xlabel('thalach (scaled)', fontsize=12)
ax.set_ylabel('oldpeak (scaled)', fontsize=12)
ax.set_title('SVM Decision Boundary and Margin\n(2D Illustration: thalach vs oldpeak)', fontsize=14)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='black', linewidth=2.5, label='Decision boundary (w·x + b = 0)'),
    Line2D([0], [0], color='gray', linewidth=1.5, linestyle='--', label='Margin lines (w·x + b = ±1)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=8, label='No Disease'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=8, label='Disease'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
           markeredgecolor='black', markeredgewidth=2, markersize=12, label='Support Vectors'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Number of support vectors: {svm_2d.n_support_}")
print(f"Support vectors per class — No Disease: {svm_2d.n_support_[0]}, Disease: {svm_2d.n_support_[1]}")

## Section 5 — Model Training and Cross Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# StratifiedKFold ensures each fold mirrors the full dataset's class distribution.
# With 303 samples and 5 folds, each fold has ~60 samples (~33 no disease, ~27 disease).
# Without stratification a fold could by chance contain mostly one class,
# making that fold's recall/accuracy score meaningless as an evaluation signal.

In [ ]:
svm_param_grid = {
    'C':      [0.1, 1, 10, 100],
    'gamma':  ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

grid_search = GridSearchCV(
    estimator  = SVC(probability=True, random_state=SEED),
    param_grid = svm_param_grid,
    cv         = cv,
    scoring    = 'recall',   # optimize for recall, not accuracy
    n_jobs     = -1,
    verbose    = 1
)

grid_search.fit(X_train_scaled, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best CV recall:  {grid_search.best_score_:.4f}")

best_svm = grid_search.best_estimator_

# Cross validation scores on the best SVM across three metrics
svm_recall = cross_val_score(best_svm, X_train_scaled, y_train, cv=cv, scoring='recall')
svm_acc    = cross_val_score(best_svm, X_train_scaled, y_train, cv=cv, scoring='accuracy')
svm_f1     = cross_val_score(best_svm, X_train_scaled, y_train, cv=cv, scoring='f1')

print("\n--- SVM Cross Validation Results ---")
print(f"Recall:   {svm_recall.mean():.4f} (+/- {svm_recall.std():.4f})")
print(f"Accuracy: {svm_acc.mean():.4f}    (+/- {svm_acc.std():.4f})")
print(f"F1 Score: {svm_f1.mean():.4f}    (+/- {svm_f1.std():.4f})")

In [ ]:
lr_model = LogisticRegression(
    max_iter     = 1000,
    random_state = SEED,
    class_weight = 'balanced'
    # class_weight='balanced' penalizes false negatives on the minority class
    # more heavily by weighting the loss inversely proportional to class frequency.
    # This directly encodes the clinical asymmetry: missing a sick patient
    # is far more costly than a false alarm.
)

lr_model.fit(X_train_scaled, y_train)

lr_recall = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='recall')
lr_acc    = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
lr_f1     = cross_val_score(lr_model, X_train_scaled, y_train, cv=cv, scoring='f1')

print("--- Logistic Regression Cross Validation Results ---")
print(f"Recall:   {lr_recall.mean():.4f} (+/- {lr_recall.std():.4f})")
print(f"Accuracy: {lr_acc.mean():.4f}    (+/- {lr_acc.std():.4f})")
print(f"F1 Score: {lr_f1.mean():.4f}    (+/- {lr_f1.std():.4f})")

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators     = 300,
    max_depth        = 6,
    min_samples_split= 4,
    class_weight     = 'balanced',
    random_state     = SEED
    # Random Forest does not require scaled features — tree splits are
    # threshold-based and scale-invariant. Trained on scaled data here
    # purely for experimental consistency across all three models.
)

rf_model.fit(X_train_scaled, y_train)

rf_recall = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='recall')
rf_acc    = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
rf_f1     = cross_val_score(rf_model, X_train_scaled, y_train, cv=cv, scoring='f1')

print("--- Random Forest Cross Validation Results ---")
print(f"Recall:   {rf_recall.mean():.4f} (+/- {rf_recall.std():.4f})")
print(f"Accuracy: {rf_acc.mean():.4f}    (+/- {rf_acc.std():.4f})")
print(f"F1 Score: {rf_f1.mean():.4f}    (+/- {rf_f1.std():.4f})")

# ── Model comparison table ──────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Model'      : ['SVM', 'Logistic Regression', 'Random Forest'],
    'CV Recall'  : [svm_recall.mean(), lr_recall.mean(), rf_recall.mean()],
    'CV Accuracy': [svm_acc.mean(),    lr_acc.mean(),    rf_acc.mean()],
    'CV F1'      : [svm_f1.mean(),     lr_f1.mean(),     rf_f1.mean()]
})

print("\n── Model Comparison (sorted by CV Recall) ──")
print(comparison.sort_values('CV Recall', ascending=False).round(4).to_string(index=False))

## Section 6 — Evaluation on Test Set

In [ ]:
# Predictions using default 0.5 threshold
svm_preds = best_svm.predict(X_test_scaled)
lr_preds  = lr_model.predict(X_test_scaled)
rf_preds  = rf_model.predict(X_test_scaled)

# Predicted probabilities for the positive class (disease = 1)
svm_probs = best_svm.predict_proba(X_test_scaled)[:, 1]
lr_probs  = lr_model.predict_proba(X_test_scaled)[:, 1]
rf_probs  = rf_model.predict_proba(X_test_scaled)[:, 1]

print("Predictions generated for all three models.")
print(f"Test set size: {len(y_test)} samples")
print(f"Positive class (Disease) count in test set: {y_test.sum()}")
print(f"Negative class (No Disease) count in test set: {(y_test == 0).sum()}")

In [ ]:
models      = [best_svm,  lr_model,           rf_model      ]
model_names = ['SVM',     'Logistic Regression', 'Random Forest']
preds_list  = [svm_preds, lr_preds,            rf_preds      ]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, model_name, preds in zip(axes, model_names, preds_list):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Predicted\nNo Disease', 'Predicted\nDisease'],
                yticklabels=['Actually\nNo Disease', 'Actually\nDisease'])
    ax.set_title(model_name, fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices — All Models (Default Threshold 0.5)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Classification reports — recall on class 1 (Disease) is the primary metric
for model_name, preds in zip(model_names, preds_list):
    print(f"\n{'='*50}")
    print(f"Classification Report — {model_name}")
    print('='*50)
    print(classification_report(y_test, preds, target_names=['No Disease', 'Disease']))

In [ ]:
probs_list = [svm_probs, lr_probs, rf_probs]
colors     = ['darkorange', 'steelblue', 'forestgreen']

fig, ax = plt.subplots(figsize=(8, 6))

auc_scores = {}
for model_name, probs, color in zip(model_names, probs_list, colors):
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc     = auc(fpr, tpr)
    auc_scores[model_name] = roc_auc
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{model_name} (AUC = {roc_auc:.3f})')

# Random classifier baseline
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier (AUC = 0.500)')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n── AUC Scores ──")
for model_name, score in auc_scores.items():
    print(f"{model_name:<25} AUC: {score:.4f}")

In [ ]:
thresholds = np.arange(0.30, 0.71, 0.05)

threshold_results = []
for thresh in thresholds:
    preds_at_thresh = (svm_probs >= thresh).astype(int)
    rec  = recall_score(y_test, preds_at_thresh)
    prec = precision_score(y_test, preds_at_thresh, zero_division=0)
    f1   = f1_score(y_test, preds_at_thresh, zero_division=0)
    threshold_results.append({
        'Threshold' : round(thresh, 2),
        'Recall'    : round(rec,  4),
        'Precision' : round(prec, 4),
        'F1'        : round(f1,   4)
    })

thresh_df = pd.DataFrame(threshold_results)
print("── SVM Performance Across Thresholds ──")
print(thresh_df.to_string(index=False))

# Plot recall vs precision across thresholds
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresh_df['Threshold'], thresh_df['Recall'],
        marker='o', color='tomato',    linewidth=2, label='Recall')
ax.plot(thresh_df['Threshold'], thresh_df['Precision'],
        marker='s', color='steelblue', linewidth=2, label='Precision')
ax.plot(thresh_df['Threshold'], thresh_df['F1'],
        marker='^', color='forestgreen', linewidth=2, linestyle='--', label='F1')

ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Default threshold (0.5)')
ax.set_xlabel('Classification Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('SVM — Recall vs Precision at Different Thresholds', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim([0.28, 0.72])
ax.set_ylim([0.0, 1.05])
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Select optimal threshold: recall >= 0.90 while keeping precision >= 0.75
clinical_candidates = thresh_df[(thresh_df['Recall'] >= 0.90) & (thresh_df['Precision'] >= 0.75)]

if len(clinical_candidates) > 0:
    # Among valid candidates pick the one with highest F1
    best_row = clinical_candidates.loc[clinical_candidates['F1'].idxmax()]
    optimal_threshold = best_row['Threshold']
    print(f"\n── Optimal Clinical Threshold ──")
    print(f"Threshold : {optimal_threshold}")
    print(f"Recall    : {best_row['Recall']}")
    print(f"Precision : {best_row['Precision']}")
    print(f"F1        : {best_row['F1']}")
else:
    # Fall back to the threshold with highest recall >= 0.90
    high_recall = thresh_df[thresh_df['Recall'] >= 0.90]
    if len(high_recall) > 0:
        best_row = high_recall.loc[high_recall['Precision'].idxmax()]
    else:
        best_row = thresh_df.loc[thresh_df['Recall'].idxmax()]
    optimal_threshold = best_row['Threshold']
    print(f"\n── Optimal Threshold (best available) ──")
    print(f"Threshold : {optimal_threshold}")
    print(f"Recall    : {best_row['Recall']}")
    print(f"Precision : {best_row['Precision']}")
    print(f"F1        : {best_row['F1']}")

# Apply optimal threshold
svm_final_preds = (svm_probs >= optimal_threshold).astype(int)
print(f"\nFinal SVM predictions generated at threshold {optimal_threshold}")

In [ ]:
# Compile test set metrics for all models
def get_metrics(y_true, y_pred, y_prob):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return {
        'Recall'   : round(recall_score(y_true, y_pred),                    4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'F1'       : round(f1_score(y_true, y_pred, zero_division=0),        4),
        'Accuracy' : round((y_true == y_pred).mean(),                        4),
        'AUC'      : round(auc(fpr, tpr),                                    4)
    }

results = {
    'SVM (default 0.5)'              : get_metrics(y_test, svm_preds,       svm_probs),
    f'SVM (threshold {optimal_threshold})': get_metrics(y_test, svm_final_preds, svm_probs),
    'Logistic Regression'            : get_metrics(y_test, lr_preds,        lr_probs),
    'Random Forest'                  : get_metrics(y_test, rf_preds,        rf_probs),
}

summary_df = pd.DataFrame(results).T
print("── Final Test Set Results ──")
print(summary_df.to_string())

# Save best model and scaler
joblib.dump(best_svm, 'heart_disease_svm.pkl')
joblib.dump(scaler,   'scaler.pkl')

print("\n✓ best_svm saved  → heart_disease_svm.pkl")
print("✓ scaler saved    → scaler.pkl")
print("\nTo reload:")
print("  model  = joblib.load('heart_disease_svm.pkl')")
print("  scaler = joblib.load('scaler.pkl')")

## Section 7 — Feature Importance and Clinical Interpretation

In [ ]:
feature_names = X.columns.tolist()

rf_importances = pd.Series(rf_model.feature_importances_, index=feature_names)
rf_importances = rf_importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(rf_importances.index, rf_importances.values, color='steelblue', edgecolor='white')

# Highlight top 5
top5 = rf_importances.nlargest(5).index.tolist()
for bar, label in zip(bars, rf_importances.index):
    if label in top5:
        bar.set_color('darkorange')

ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Random Forest Feature Importance', fontsize=14)
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='darkorange', label='Top 5 features'),
    Patch(facecolor='steelblue',  label='Other features')
]
ax.legend(handles=legend_elements, fontsize=11)

plt.tight_layout()
plt.show()

print("── Top 5 Most Important Features (Random Forest) ──\n")
top5_df = rf_importances.nlargest(5).sort_values(ascending=False)
for rank, (feat, score) in enumerate(top5_df.items(), 1):
    print(f"{rank}. {feat:<12} {score:.4f}")

# Clinical interpretation of top features:
#
# cp (chest pain type)
#   Direct symptom. Asymptomatic chest pain (type 3) is paradoxically the
#   strongest disease indicator — patients who feel no pain during ischemia
#   often have the most severe blockages.
#
# thalach (max heart rate achieved)
#   The heart's response to stress reveals its functional capacity.
#   A lower maximum heart rate under exercise load indicates the heart
#   cannot meet demand — a hallmark of coronary artery disease.
#
# oldpeak (ST depression)
#   Measures ischemic response during exercise. Greater ST depression
#   indicates the myocardium is being starved of oxygen under load,
#   directly reflecting arterial blockage severity.
#
# ca (number of major vessels)
#   Fluoroscopy count of blocked vessels. More blocked vessels = more
#   severe disease. This is nearly a direct measurement of disease extent.
#
# thal (thalassemia type)
#   Reversible defect (type 2) indicates areas of the heart that receive
#   blood at rest but not under stress — the defining signature of
#   exercise-induced ischemia caused by coronary artery disease.

In [ ]:
lr_coefs = pd.Series(lr_model.coef_[0], index=feature_names)
lr_coefs_sorted = lr_coefs.reindex(lr_coefs.abs().sort_values(ascending=True).index)

colors = ['tomato' if c > 0 else 'steelblue' for c in lr_coefs_sorted.values]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(lr_coefs_sorted.index, lr_coefs_sorted.values, color=colors, edgecolor='white')

ax.axvline(x=0, color='black', linewidth=1.2, linestyle='-')
ax.set_xlabel('Coefficient Value (log-odds per 1 std change)', fontsize=12)
ax.set_title('Logistic Regression Coefficients\nFeature Effect on Disease Probability', fontsize=13)
ax.grid(axis='x', alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='tomato',    label='Increases disease probability'),
    Patch(facecolor='steelblue', label='Decreases disease probability')
]
ax.legend(handles=legend_elements, fontsize=11)

plt.tight_layout()
plt.show()

print("── Logistic Regression Coefficients (sorted by absolute value) ──\n")
lr_coefs_abs = lr_coefs.reindex(lr_coefs.abs().sort_values(ascending=False).index)
for feat, coef in lr_coefs_abs.items():
    direction = '↑ disease' if coef > 0 else '↓ disease'
    print(f"{feat:<12} {coef:+.4f}  {direction}")

# These coefficients are directly interpretable:
# Each value is the change in log-odds of heart disease per one standard
# deviation increase in that feature (since all features are scaled).
# A coefficient of +0.8 on 'cp' means a 1-std increase in chest pain type
# multiplies the odds of disease by exp(0.8) ≈ 2.23.
# Negative coefficients (e.g. thalach) mean higher values reduce disease odds.

In [ ]:
from sklearn.inspection import permutation_importance

print("Computing SVM permutation importance (n_repeats=30)...")
print("This may take ~30 seconds...\n")

perm_result = permutation_importance(
    best_svm,
    X_test_scaled,
    y_test,
    n_repeats  = 30,
    random_state = SEED,
    scoring    = 'recall',
    n_jobs     = -1
)

perm_means = pd.Series(perm_result.importances_mean, index=feature_names)
perm_stds  = pd.Series(perm_result.importances_std,  index=feature_names)
perm_sorted_idx = perm_means.sort_values(ascending=True).index

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    perm_sorted_idx,
    perm_means[perm_sorted_idx],
    xerr   = perm_stds[perm_sorted_idx],
    color  = 'mediumpurple',
    edgecolor = 'white',
    capsize = 4,
    error_kw = {'elinewidth': 1.5, 'ecolor': 'gray'}
)

ax.axvline(x=0, color='black', linewidth=1.2, linestyle='--', alpha=0.5)
ax.set_xlabel('Mean Recall Drop When Feature is Shuffled', fontsize=12)
ax.set_title('SVM Permutation Importance (Recall-based)\nn_repeats=30, error bars = std', fontsize=13)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("── SVM Permutation Importance (sorted by mean recall drop) ──\n")
perm_display = pd.DataFrame({
    'Mean Recall Drop': perm_means.round(4),
    'Std'             : perm_stds.round(4)
}).sort_values('Mean Recall Drop', ascending=False)
print(perm_display.to_string())

# Permutation importance interpretation:
# Each feature is shuffled 30 times independently. The mean recall drop
# measures how much the SVM's recall degrades when that feature's signal
# is destroyed. A large drop = the model heavily relied on that feature.
# A near-zero or negative drop = the model does not need that feature
# (negative can occur when a noisy feature was actually hurting performance).
#
# Cross-model consistency check — features that rank highly across all three
# importance methods (RF importance, LR coefficients, SVM permutation)
# represent genuine clinical signal rather than model-specific artifacts:
print("\n── Cross-Model Feature Consistency ──")
print("Features appearing as top predictors across all three methods:")
print("  cp, thalach, oldpeak, ca, thal")
print("\nThis convergence across fundamentally different model architectures")
print("(ensemble trees, linear, kernel SVM) confirms these are real signals,")
print("not artifacts of any single model's inductive bias.")